# Лабораторная работа №8: Active learning и human-in-the-loop

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Реализовать random, entropy и diversity sampling.
- Провести несколько AL-итераций.
- Построить learning curves при равном бюджете.


## Теоретическая часть

Active learning выбирает следующий batch из pool. Uncertainty ищет неуверенные объекты, diversity покрывает пространство, disagreement использует расхождение моделей. Сравнение корректно только при одинаковом annotation budget.


### Математическая постановка

Entropy sampling: $u(x)=-\sum_c p(c\mid x)\log p(c\mid x)$. Diversity можно реализовать k-center: выбрать объект, максимизирующий расстояние до уже размеченного множества.


### Материалы

- [Active Learning survey](https://arxiv.org/abs/2009.00236)
- [scikit-learn calibration](https://scikit-learn.org/stable/modules/calibration.html)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Работа выполняется на изображениях рукописных цифр `sklearn.datasets.load_digits`; oracle в этом воспроизводимом эксперименте симулирует человека, открывая истинную метку только после отбора объекта. FAST_DEV_RUN — два раунда проверки кода, полный режим — восемь раундов и повторный random baseline.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))

## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Стратифицированный отложенный тест

*Вес: 1 балл.*

**Постановка.** Этап «стратифицированный отложенный тест» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Разделите изображения цифр по стратифицированному правилу до выбора стратегии.
2. Сохраните и предъявите индексы train/test изображений цифр.

**Результат.** индексы train/test изображений цифр.

**Критерий принятия.** Задание считается принятым, если индексы train/test изображений цифр получены из фактических входов этого этапа и доступны для проверки.

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X_all = digits.data.astype(np.float32) / 16.0          # 8×8 изображения, значения 0..16 → 0..1
y_all = digits.target
idx_all = np.arange(len(y_all))
# test откладывается до выбора стратегии и больше не меняется
train_idx, test_idx = train_test_split(idx_all, test_size=0.3, stratify=y_all, random_state=SEED)
train_idx, test_idx = np.sort(train_idx), np.sort(test_idx)
X_train, y_train, X_test, y_test = X_all[train_idx], y_all[train_idx], X_all[test_idx], y_all[test_idx]
assert not set(train_idx) & set(test_idx)
print(f"изображений: {len(y_all)}; train (pool + initial): {len(train_idx)}, test: {len(test_idx)}")
print("test по классам:", np.bincount(y_test).tolist())

изображений: 1797; train (pool + initial): 1257, test: 540
test по классам: [54, 55, 53, 55, 54, 55, 54, 54, 52, 54]


### Начальная размеченная выборка

*Вес: 1 балл.*

**Постановка.** Этап «начальная размеченная выборка» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Задайте initial с каждым классом и неразмеченный pool только из train.
2. Сохраните и предъявите непересекающиеся initial и pool.

**Результат.** непересекающиеся initial и pool.

**Критерий принятия.** Задание считается принятым, если непересекающиеся initial и pool получены из фактических входов этого этапа и доступны для проверки.

In [3]:
INIT_PER_CLASS = 2
rng = np.random.RandomState(SEED)
initial = np.concatenate([rng.choice(np.where(y_train == c)[0], INIT_PER_CLASS, replace=False) for c in range(10)])
pool = np.setdiff1d(np.arange(len(train_idx)), initial)          # позиции внутри train
assert not set(initial) & set(pool) and len(initial) + len(pool) == len(train_idx)
assert set(y_train[initial]) == set(range(10))
print(f"initial: {len(initial)} объектов (по {INIT_PER_CLASS} на класс), неразмеченный pool: {len(pool)}")

initial: 20 объектов (по 2 на класс), неразмеченный pool: 1237


### Протокол симулированного разметчика

*Вес: 1 балл.*

**Постановка.** Этап «протокол симулированного разметчика» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните split и документируйте, что oracle открывает метку лишь после выбора.
2. Сохраните и предъявите artifacts/split.json с происхождением меток.

**Результат.** artifacts/split.json с происхождением меток.

**Критерий принятия.** Задание считается принятым, если artifacts/split.json с происхождением меток получены из фактических входов этого этапа и доступны для проверки.

In [4]:
BATCH = 20
ROUNDS = 2 if FAST_DEV_RUN else 8
split_info = {"dataset": "sklearn.datasets.load_digits (UCI ML hand-written digits, 8x8)", "license": "CC BY 4.0 (UCI)",
              "seed": SEED, "train_idx": train_idx.tolist(), "test_idx": test_idx.tolist(),
              "initial_train_positions": initial.tolist(), "pool_size": int(len(pool)), "batch_per_round": BATCH, "rounds": ROUNDS,
              "label_source": {"initial": "simulated_oracle", "queried": "simulated_oracle"},
              "oracle_protocol": ("Человек-разметчик симулируется oracle: истинная метка объекта из pool открывается "
                                  "только после того, как стратегия выбрала этот объект. Стратегии видят лишь признаки pool. "
                                  "Это воспроизводимая симуляция, а не ручная разметка.")}
save_json("split.json", split_info)
print(split_info["oracle_protocol"])

Человек-разметчик симулируется oracle: истинная метка объекта из pool открывается только после того, как стратегия выбрала этот объект. Стратегии видят лишь признаки pool. Это воспроизводимая симуляция, а не ручная разметка.


### Три стратегии запроса

*Вес: 1 балл.*

**Постановка.** Этап «три стратегии запроса» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Реализуйте случайный, энтропийный и k-center выбор; в k-center обновляйте расстояния после каждого объекта.
2. Сохраните и предъявите выбор random, entropy и k-center.

**Результат.** выбор random, entropy и k-center.

**Критерий принятия.** Задание считается принятым, если выбор random, entropy и k-center получены из фактических входов этого этапа и доступны для проверки.

In [5]:
from scipy.special import entr

class Oracle:
    """Хранит истинные метки pool и выдаёт их только по запросу; ведёт журнал запросов."""
    def __init__(self, labels):
        self._labels = labels; self.log = []
    def reveal(self, positions, round_id, strategy):
        out = []
        for p in positions:
            self.log.append({"train_position": int(p), "round": round_id, "strategy": strategy, "source": "simulated_oracle"})
            out.append(int(self._labels[p]))
        return out

def select_random(model, X_pool_pos, labeled_pos, k, rng):
    return list(rng.choice(X_pool_pos, k, replace=False))

def select_entropy(model, X_pool_pos, labeled_pos, k, rng):
    probs = model.predict_proba(X_train[X_pool_pos])
    u = entr(probs).sum(1)                                   # −Σ p log p
    return list(np.asarray(X_pool_pos)[np.argsort(-u, kind="stable")[:k]])

def select_kcenter(model, X_pool_pos, labeled_pos, k, rng):
    """Greedy k-center: каждый раз берём объект, наиболее удалённый от уже размеченных и уже выбранных."""
    cand = np.asarray(X_pool_pos)
    d = np.min(np.linalg.norm(X_train[cand][:, None] - X_train[labeled_pos][None], axis=2), axis=1)
    chosen = []
    for _ in range(k):
        j = int(np.argmax(d))
        chosen.append(int(cand[j]))
        d = np.minimum(d, np.linalg.norm(X_train[cand] - X_train[cand[j]], axis=1))   # пересчёт после каждого объекта
        d[j] = -1
    return chosen

STRATEGIES = {"random": select_random, "entropy": select_entropy, "kcenter": select_kcenter}
print("стратегии:", list(STRATEGIES))

стратегии: ['random', 'entropy', 'kcenter']


### Итерационное переобучение

*Вес: 1 балл.*

**Постановка.** Этап «итерационное переобучение» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. После каждого запроса дообучайте классификатор на открытых метках и фиксируйте trace.
2. Сохраните и предъявите функция run с фактическими запрошенными метками.

**Результат.** функция run с фактическими запрошенными метками.

**Критерий принятия.** Задание считается принятым, если функция run с фактическими запрошенными метками получены из фактических входов этого этапа и доступны для проверки.

In [6]:
from sklearn.linear_model import LogisticRegression

def make_model():
    return LogisticRegression(C=1.0, max_iter=3000)

def run(strategy, seed=SEED, rounds=ROUNDS, init=initial):
    rng = np.random.RandomState(seed)
    oracle = Oracle(y_train)
    labeled = list(init)
    labels = oracle.reveal(init, 0, "initial")
    init_set = set(int(p) for p in init)
    unlabeled = [p for p in range(len(train_idx)) if p not in init_set]
    trace = []
    for r in range(rounds + 1):
        model = make_model().fit(X_train[labeled], labels)   # переобучение на всех открытых метках
        acc = float((model.predict(X_test) == y_test).mean())
        trace.append({"round": r, "n_labeled": len(labeled), "test_accuracy": acc})
        if r == rounds:
            break
        picked = STRATEGIES[strategy](model, unlabeled, labeled, BATCH, rng)
        labels += oracle.reveal(picked, r + 1, strategy)
        labeled += picked
        unlabeled = [p for p in unlabeled if p not in set(picked)]
    assert len(set(labeled)) == len(labeled), "объект запрошен дважды"
    return {"strategy": strategy, "seed": seed, "trace": trace, "queried": oracle.log, "final_model": model,
            "labeled": labeled, "class_counts": np.bincount(np.array(labels), minlength=10).tolist()}

main_runs = {s: run(s) for s in STRATEGIES}
for s, res in main_runs.items():
    print(f"{s:8s}", [round(t["test_accuracy"], 3) for t in res["trace"]], "| метки по классам:", res["class_counts"])

random   [0.719, 0.78, 0.815, 0.872, 0.881, 0.906, 0.902, 0.915, 0.926] | метки по классам: [14, 19, 19, 23, 18, 17, 16, 17, 18, 19]
entropy  [0.719, 0.428, 0.624, 0.82, 0.841, 0.894, 0.9, 0.92, 0.926] | метки по классам: [11, 15, 19, 19, 13, 21, 11, 12, 34, 25]
kcenter  [0.719, 0.846, 0.894, 0.902, 0.919, 0.93, 0.937, 0.937, 0.941] | метки по классам: [8, 14, 18, 20, 18, 21, 10, 19, 31, 21]


### Кривые при общем бюджете

*Вес: 1 балл.*

**Постановка.** Этап «кривые при общем бюджете» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Вычислите точки кривых для одинакового числа запросов и сохраните бюджеты.
2. Сохраните и предъявите artifacts/learning_curves.json.

**Результат.** artifacts/learning_curves.json.

**Критерий принятия.** Задание считается принятым, если artifacts/learning_curves.json получены из фактических входов этого этапа и доступны для проверки.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/learning_curves.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Интеграл по бюджету разметки

*Вес: 1 балл.*

**Постановка.** Этап «интеграл по бюджету разметки» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Проинтегрируйте кривые accuracy по числу полученных меток.
2. Сохраните и предъявите AUC и конечная accuracy из фактических кривых.

**Результат.** AUC и конечная accuracy из фактических кривых.

**Критерий принятия.** Задание считается принятым, если AUC и конечная accuracy из фактических кривых получены из фактических входов этого этапа и доступны для проверки.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: AUC и конечная accuracy из фактических кривых
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Повторные случайные запросы

*Вес: 1 балл.*

**Постановка.** Этап «повторные случайные запросы» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Повторите random с разными генераторами при неизменном train/test.
2. Сохраните и предъявите результаты random при различных seed.

**Результат.** результаты random при различных seed.

**Критерий принятия.** Задание считается принятым, если результаты random при различных seed получены из фактических входов этого этапа и доступны для проверки.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: результаты random при различных seed
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Разбор ошибок отложенного теста

*Вес: 1 балл.*

**Постановка.** Этап «разбор ошибок отложенного теста» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сохраните изображения и истинные/предсказанные классы ошибочных test-примеров.
2. Сохраните и предъявите artifacts/test_errors.json с неверно распознанными изображениями.

**Результат.** artifacts/test_errors.json с неверно распознанными изображениями.

**Критерий принятия.** Задание считается принятым, если artifacts/test_errors.json с неверно распознанными изображениями получены из фактических входов этого этапа и доступны для проверки.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/test_errors.json с неверно распознанными изображениями
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Сводка сравнительного эксперимента

*Вес: 1 балл.*

**Постановка.** Этап «сводка сравнительного эксперимента» необходим для воспроизводимого предметного сравнения. Результат этого шага нельзя заменить синтетической оценкой или произвольной метрикой.

Выполните следующие действия:

1. Сопоставьте AUC и конечную точность по трём стратегиям.
2. Сохраните и предъявите artifacts/comparison.json.

**Результат.** artifacts/comparison.json.

**Критерий принятия.** Задание считается принятым, если artifacts/comparison.json получены из фактических входов этого этапа и доступны для проверки.

In [ ]:
# Реализуйте это задание по постановке выше.
# Артефакт: artifacts/comparison.json
raise NotImplementedError("Реализуйте это задание в этой ячейке")

### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [ ]:
# Реализуйте творческое мини-исследование по постановке выше.
raise NotImplementedError("Реализуйте бонусное исследование")

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.